In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
#cell 1

import os, joblib, numpy as np, pandas as pd, torch
from pathlib import Path
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.model_selection import StratifiedKFold
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# === EDIT THESE THREE PATH ROOTS ONLY ===
DATA_ROOT   = "/kaggle/input/fake-or-real-the-impostor-hunt/data"
TFIDF_DIR   = "/kaggle/input/tfidf-export-v1"   # vectorizer.joblib + classifier.joblib
ROBERTA_DIR = "/kaggle/input/subtlety-roberta-v1/content/roberta_official_v1/best"

TRAIN_CSV = f"{DATA_ROOT}/train.csv"
TRAIN_DIR = f"{DATA_ROOT}/train"
TEST_DIR  = f"{DATA_ROOT}/test"

for k, v in {
    "train.csv" : os.path.exists(TRAIN_CSV),
    "train dir" : os.path.exists(TRAIN_DIR),
    "test dir"  : os.path.exists(TEST_DIR),
    "tfidf dir" : os.path.exists(TFIDF_DIR),
    "roberta   " : os.path.exists(ROBERTA_DIR),
}.items():
    print(f"{k:10s}: {v}")
    if not v:
        raise FileNotFoundError(f"Missing required path: {k}")

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

np.random.seed(42)
torch.manual_seed(42)

train.csv : True
train dir : True
test dir  : True
tfidf dir : True
roberta   : True
Device: cuda


In [3]:
#cell 2

# ---- load train pairs ----
def read_txt(p): 
    with open(p, "r", encoding="utf-8") as f: 
        return f.read()

idx = pd.read_csv(TRAIN_CSV)
rows = []
for _, r in idx.iterrows():
    art_id  = int(r["id"])
    real_id = int(r["real_text_id"])
    folder  = os.path.join(TRAIN_DIR, f"article_{art_id:04d}")
    t1 = read_txt(os.path.join(folder, "file_1.txt"))
    t2 = read_txt(os.path.join(folder, "file_2.txt"))
    rows.append({"id": art_id, "text1": t1, "text2": t2, "gold": real_id})
pairs = pd.DataFrame(rows)
print("Built pairs:", pairs.shape)

# ---- TF‑IDF artifacts ----
vec = joblib.load(os.path.join(TFIDF_DIR, "vectorizer.joblib"))
clf = joblib.load(os.path.join(TFIDF_DIR, "classifier.joblib"))
X1 = vec.transform(pairs["text1"])
X2 = vec.transform(pairs["text2"])
P1 = clf.predict_proba(X1); P2 = clf.predict_proba(X2)

def tfidf_acc_for(col):
    pred = np.where(P1[:, col] >= P2[:, col], 1, 2)
    return accuracy_score(pairs["gold"], pred)

acc0, acc1 = tfidf_acc_for(0), tfidf_acc_for(1)
tfidf_real_col = 0 if acc0 >= acc1 else 1
print(f"TF‑IDF mapping: real_class_index={tfidf_real_col} (cal acc={max(acc0,acc1):.3f})")

def tfidf_probs(df):
    A = vec.transform(df["text1"]); B = vec.transform(df["text2"])
    return clf.predict_proba(A)[:, tfidf_real_col], clf.predict_proba(B)[:, tfidf_real_col]

# ---- RoBERTa (local, no internet) ----
tok = AutoTokenizer.from_pretrained(ROBERTA_DIR, local_files_only=True)
rob = AutoModelForSequenceClassification.from_pretrained(ROBERTA_DIR, local_files_only=True).to(device).eval()

@torch.no_grad()
def roberta_probs_batch(t1_list, t2_list, max_len=384, batch_size=16):
    outs = []
    for i in range(0, len(t1_list), batch_size):
        enc = tok(
            t1_list[i:i+batch_size], t2_list[i:i+batch_size],
            truncation=True, max_length=max_len, padding=True, return_tensors="pt"
        ).to(device)
        logits = rob(**enc).logits.float()
        probs = torch.softmax(logits, dim=-1).cpu().numpy()
        outs.append(probs)
    return np.vstack(outs)

# Calibrate which logit means “text1 is real”
probs_train = roberta_probs_batch(pairs["text1"].tolist(), pairs["text2"].tolist())
def rb_acc_for(col):
    other = 1 - col
    pred = np.where(probs_train[:, col] >= probs_train[:, other], 1, 2)
    return accuracy_score(pairs["gold"], pred)

acc_col1, acc_col0 = rb_acc_for(1), rb_acc_for(0)
roberta_text1_col = 1 if acc_col1 >= acc_col0 else 0
print(f"RoBERTa mapping: 'text1 is real' -> column {roberta_text1_col} (cal acc={max(acc_col0,acc_col1):.3f})")

Built pairs: (95, 4)
TF‑IDF mapping: real_class_index=0 (cal acc=0.968)


2025-08-10 22:18:42.533803: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1754864322.708873      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1754864322.762360      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


RoBERTa mapping: 'text1 is real' -> column 0 (cal acc=0.768)


In [4]:
# cell 3

# --- Rebuild pairs fresh + fast CV over thresholds ---

import os, pandas as pd, numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score

# 0) Rebuild pairs from train.csv + train folder (overwrites any old 'pairs')
def _build_pairs(train_csv, train_dir):
    def read_txt(p):
        with open(p, "r", encoding="utf-8") as f: return f.read()
    idx = pd.read_csv(train_csv)
    rows = []
    for _, r in idx.iterrows():
        aid  = int(r["id"])
        real = int(r["real_text_id"])  # 1 or 2
        folder = os.path.join(train_dir, f"article_{aid:04d}")
        t1 = read_txt(os.path.join(folder, "file_1.txt"))
        t2 = read_txt(os.path.join(folder, "file_2.txt"))
        rows.append({"id": aid, "text1": t1, "text2": t2, "gold_real_text_id": real})
    return pd.DataFrame(rows)

pairs = _build_pairs(TRAIN_CSV, TRAIN_DIR)
labels = pairs["gold_real_text_id"].values
N = len(pairs)
print(f"Built pairs: {pairs.shape}")

# 1) Require helpers from the TF‑IDF/RoBERTa cell
for name in ["tfidf_probs", "roberta_probs_batch", "roberta_text1_col"]:
    if name not in globals():
        raise RuntimeError(f"Missing `{name}` — run the TF‑IDF/RoBERTa helpers cell first.")

# 2) Cache probabilities once (so CV is fast)
print("Precomputing TF‑IDF and RoBERTa probabilities…")
tf1_all, tf2_all = tfidf_probs(pairs)  # (N,), (N,)
rb_all = roberta_probs_batch(pairs["text1"].tolist(), pairs["text2"].tolist())  # (N,2)
rb1_all = rb_all[:, roberta_text1_col]
rb2_all = rb_all[:, 1 - roberta_text1_col]

def rule_predict_indices(idx, T_TFIDF, T_ROBERTA):
    tf1, tf2 = tf1_all[idx], tf2_all[idx]
    rb1, rb2 = rb1_all[idx], rb2_all[idx]
    tf_margin = np.abs(tf1 - tf2)
    rb_margin = np.abs(rb1 - rb2)
    # default to TF‑IDF
    pred = np.where(tf1 >= tf2, 1, 2)
    # override where TF‑IDF is unsure but RoBERTa is confident
    mask = (tf_margin < T_TFIDF) & (rb_margin >= T_ROBERTA)
    pred[mask] = np.where(rb1[mask] >= rb2[mask], 1, 2)
    return pred

# 3) Small grid search with 5‑fold CV
T_TFIDF_grid   = [0.00, 0.01, 0.02, 0.03, 0.05]
T_ROBERTA_grid = [0.40, 0.50, 0.60]

best = None; best_acc = -1; best_std = 999
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for t_tf in T_TFIDF_grid:
    for t_rb in T_ROBERTA_grid:
        accs = []
        for _, va_idx in skf.split(np.zeros(N), labels):
            pred_va = rule_predict_indices(va_idx, t_tf, t_rb)
            accs.append(accuracy_score(labels[va_idx], pred_va))
        acc, std = float(np.mean(accs)), float(np.std(accs))
        if (acc > best_acc) or (acc == best_acc and std < best_std):
            best_acc, best_std, best = acc, std, (t_tf, t_rb)

T_TFIDF_star, T_ROBERTA_star = best
print(f"Best thresholds by CV: T_TFIDF={T_TFIDF_star}, T_ROBERTA={T_ROBERTA_star} | "
      f"acc={best_acc:.4f} ± {best_std:.4f}")

# 4) Train‑set metrics with tuned thresholds
pred_train = rule_predict_indices(np.arange(N), T_TFIDF_star, T_ROBERTA_star)
print("📏 Ensemble on TRAIN -> acc: %.4f | f1: %.4f" % (
    accuracy_score(labels, pred_train),
    f1_score(labels, pred_train, average="macro")
))

Built pairs: (95, 4)
Precomputing TF‑IDF and RoBERTa probabilities…
Best thresholds by CV: T_TFIDF=0.01, T_ROBERTA=0.5 | acc=0.9789 ± 0.0258
📏 Ensemble on TRAIN -> acc: 0.9789 | f1: 0.9789


In [5]:
#cell 4

# --- Bootstrap missing globals (safe to re-run) ---
import os, joblib, numpy as np, torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# 1) Load TF-IDF artifacts if not already present
if 'vectorizer' not in globals() or 'clf' not in globals():
    TFIDF_DIR = globals().get('TFIDF_DIR', '/kaggle/input/tfidf-export-v1')
    vectorizer = joblib.load(os.path.join(TFIDF_DIR, 'vectorizer.joblib'))
    clf        = joblib.load(os.path.join(TFIDF_DIR, 'classifier.joblib'))
    print("Loaded TF‑IDF artifacts.")

# 2) Calibrate which prob column = “text1 is real” if needed
if 'tfidf_real_col' not in globals():
    assert 'pairs' in globals(), "Need 'pairs' to calibrate TF‑IDF. Run the cell that builds pairs first."
    X1, X2 = vectorizer.transform(pairs['text1']), vectorizer.transform(pairs['text2'])
    P1, P2 = clf.predict_proba(X1), clf.predict_proba(X2)
    def acc_for(col):
        pred = np.where(P1[:, col] >= P2[:, col], 1, 2)
        return (pred == pairs['gold_real_text_id'].values).mean()
    acc0, acc1 = acc_for(0), acc_for(1)
    tfidf_real_col = 0 if acc0 >= acc1 else 1
    print(f"Calibrated tfidf_real_col={tfidf_real_col} (acc0={acc0:.3f}, acc1={acc1:.3f})")

# 3) Load RoBERTa artifacts if not present
if 'tok_rb' not in globals() or 'rob' not in globals():
    ROBERTA_DIR = globals().get('ROBERTA_DIR', '/kaggle/input/subtlety-roberta-v1/content/roberta_official_v1/best')
    device = "cuda" if torch.cuda.is_available() else "cpu"
    tok_rb = AutoTokenizer.from_pretrained(ROBERTA_DIR)
    rob    = AutoModelForSequenceClassification.from_pretrained(ROBERTA_DIR).to(device).eval()
    print("Loaded RoBERTa model/tokenizer.")
# --- end bootstrap ---

# === SUBMIT: score TRAIN (sanity) + build TEST submission ===
import os, numpy as np, pandas as pd
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import torch

# ---- thresholds: use tuned if present, else defaults ----
T_TFIDF   = float(globals().get("T_TFIDF_star", 0.01))
T_ROBERTA = float(globals().get("T_ROBERTA_star", 0.50))
print(f"Using thresholds -> T_TFIDF={T_TFIDF}, T_ROBERTA={T_ROBERTA}")

device = device if "device" in globals() else ("cuda" if torch.cuda.is_available() else "cpu")

# ---- helpers (use already-loaded models/artifacts) ----
def tfidf_pair_probs(df):
    X1 = vectorizer.transform(df["text1"])
    X2 = vectorizer.transform(df["text2"])
    p1 = clf.predict_proba(X1)[:, tfidf_real_col]
    p2 = clf.predict_proba(X2)[:, tfidf_real_col]
    return p1, p2

@torch.no_grad()
def roberta_pair_probs(df, max_len=384, batch=16):
    probs = []
    for i in range(0, len(df), batch):
        t1 = df["text1"].iloc[i:i+batch].tolist()
        t2 = df["text2"].iloc[i:i+batch].tolist()
        enc = tok_rb(t1, t2, truncation=True, max_length=max_len,
                     padding=True, return_tensors="pt").to(device)
        logits = rob(**enc).logits.float()
        p = torch.softmax(logits, dim=-1).cpu().numpy()
        probs.append(p)
    P = np.vstack(probs)
    # we previously calibrated that column 0 corresponds to "text1 is real"
    rb_col = globals().get("roberta_text1_col", 0)
    return P[:, rb_col], P[:, 1 - rb_col]

def ensemble_predict(df, Ttf=0.01, Trb=0.50):
    tf1, tf2 = tfidf_pair_probs(df)
    rb1, rb2 = roberta_pair_probs(df)

    tf_margin = np.abs(tf1 - tf2)
    rb_margin = np.abs(rb1 - rb2)

    tf_pred = np.where(tf1 >= tf2, 1, 2)
    rb_pred = np.where(rb1 >= rb2, 1, 2)

    # rule: trust TF-IDF if confident; else let RoBERTa override if confident; else fallback TF-IDF
    ens = []
    overrides = 0
    for i in range(len(df)):
        if tf_margin[i] >= Ttf:
            ens.append(tf_pred[i])
        elif rb_margin[i] >= Trb:
            ens.append(rb_pred[i]); overrides += 1
        else:
            ens.append(tf_pred[i])
    ens = np.array(ens)

    stats = {
        "override_count": overrides,
        "override_rate": overrides / max(1, len(df)),
        "tfidf_margin_mean_all": float(tf_margin.mean()),
        "roberta_margin_mean_all": float(rb_margin.mean()),
        "tfidf_margin_mean_override": float(tf_margin[rb_margin >= Trb].mean()) if np.any(rb_margin >= Trb) else 0.0,
        "roberta_margin_mean_override": float(rb_margin[rb_margin >= Trb].mean()) if np.any(rb_margin >= Trb) else 0.0,
    }
    return ens, tf_pred, rb_pred, stats

# ---- 1) Sanity-check on TRAIN ----
if "pairs" in globals() and "gold_real_text_id" in pairs.columns:
    ens_train, tf_train, rb_train, st_train = ensemble_predict(pairs, T_TFIDF, T_ROBERTA)
    y = pairs["gold_real_text_id"].values
    acc = accuracy_score(y, ens_train)
    f1  = f1_score(y, ens_train, average="macro")
    print("\n📏 Ensemble on TRAIN (sanity check)")
    print(f"Accuracy: {acc:.4f} | F1: {f1:.4f}")
    print(f"Overrides: {st_train['override_count']} ({100*st_train['override_rate']:.1f}%)")
    print(f"Avg TF-IDF margin (all): {st_train['tfidf_margin_mean_all']:.4f}")
    print(f"Avg RoBERTa margin (all): {st_train['roberta_margin_mean_all']:.4f}")
else:
    print("\n(Train sanity skipped: 'pairs' not found)")

# ---- 2) Build TEST pairs and write submission ----
def load_test_pairs(test_dir):
    ids, t1s, t2s = [], [], []
    for name in sorted(os.listdir(test_dir)):
        if not name.startswith("article_"): 
            continue
        art_id = int(name.split("_")[1])
        folder = os.path.join(test_dir, name)
        with open(os.path.join(folder, "file_1.txt"), "r", encoding="utf-8") as f:
            t1 = f.read()
        with open(os.path.join(folder, "file_2.txt"), "r", encoding="utf-8") as f:
            t2 = f.read()
        ids.append(art_id); t1s.append(t1); t2s.append(t2)
    return pd.DataFrame({"id": ids, "text1": t1s, "text2": t2s}).sort_values("id").reset_index(drop=True)

print("\n📝 Loading TEST pairs…")
test_pairs = load_test_pairs(TEST_DIR)
print(f"Loaded {len(test_pairs)} test pairs.")

ens_test, tf_test, rb_test, st_test = ensemble_predict(test_pairs, T_TFIDF, T_ROBERTA)

sub = pd.DataFrame({"id": test_pairs["id"], "real_text_id": ens_test})
out_path = "/kaggle/working/submission.csv"
sub.to_csv(out_path, index=False)

print(f"\n✅ Submission written to: {out_path}")
print(f"Overrides on TEST: {st_test['override_count']} ({100*st_test['override_rate']:.1f}%)")

Loaded TF‑IDF artifacts.
Loaded RoBERTa model/tokenizer.
Using thresholds -> T_TFIDF=0.01, T_ROBERTA=0.5

📏 Ensemble on TRAIN (sanity check)
Accuracy: 0.9789 | F1: 0.9789
Overrides: 6 (6.3%)
Avg TF-IDF margin (all): 0.0501
Avg RoBERTa margin (all): 0.4584

📝 Loading TEST pairs…
Loaded 1068 test pairs.

✅ Submission written to: /kaggle/working/submission.csv
Overrides on TEST: 37 (3.5%)
